# Pipeline examples

Dataset source: https://www.kaggle.com/datasets/heartzhacker/medical-imaging

In [20]:
from hyppopipe.data import ImageFolderDataset, split_random_fractions
from hyppopipe.data.image import Image
from hyppopipe.pipeline import Pipeline, Step
from hyppopipe.pipeline.image import ImageTransformer, ImageClassifier
from hyppopipe.train import Trainer, TrainingConfig, ModelCandidate
from hyppopipe.train.objectives import ClassificationObjectives
from torchvision.models import resnet18, ResNet18_Weights, efficientnet_b0, EfficientNet_B0_Weights

In [ ]:
dataset = ImageFolderDataset(
    root="../datasets/Medical-imaging-dataset"
)

In [4]:
len(dataset)

4000

In [6]:
fgds_split = split_random_fractions(dataset, (0.8, 0.2))

In [8]:
print(len(fgds_split.train), len(fgds_split.val))

3200 800


In [17]:
fgds_pipeline = Pipeline(
    {
        "transform": Step(
            ImageTransformer().sharpen(),
        ),
        "classify": Step(
            ImageClassifier(num_classes=4)
        )
    },
)

In [25]:
train_result = fgds_pipeline.train(
    {
        "classify": Trainer(
            model_candidates=[
                ModelCandidate(
                    resnet18,
                    weights=[
                        ResNet18_Weights.IMAGENET1K_V1,
                    ],
                ),
                ModelCandidate(
                    efficientnet_b0,
                    weights=[
                        EfficientNet_B0_Weights.IMAGENET1K_V1,
                    ],
                ),
            ],
            monitor=ClassificationObjectives.monitor("recall"),
        ),
    },
    data=fgds_split,
    config=TrainingConfig(
        epochs=5,
        batch_size=16,
    ),
)

19:00:02 INFO hyppopipe.pipeline.pipeline: Pipeline training started for steps: classify
19:00:02 INFO hyppopipe.train.trainer: Step 'classify': starting (ImageClassifier); train=3200 val=800 epochs=5 batch_size=16 val_batch_size=16 device=mps
19:00:14 INFO hyppopipe.train.trainer: Step 'classify' model 'resnet18_IMAGENET1K_V1': training on mps (5 epochs, early_stopping=on, monitor=recall_macro)
classify · resnet18_IMAGENET1K_V1: 100%|██████████| 5/5 [02:57<00:00, 35.55s/epoch, recall_macro=0.9371, t=36.0s, train=0.4001, val=0.2063]
19:03:12 INFO hyppopipe.train.trainer: Step 'classify' resnet18_IMAGENET1K_V1: done in 189.4s — epochs=5/5 checkpoint_score=0.937079 train_loss=0.400112 val_loss=0.206256 checkpoint=/Users/nemo/diploma/Georgii-Lanin-gmlanin/examples/classify_resnet18_IMAGENET1K_V1_best.pth
19:03:23 INFO hyppopipe.train.trainer: Step 'classify' model 'efficientnet_b0_IMAGENET1K_V1': training on mps (5 epochs, early_stopping=on, monitor=recall_macro)
classify · efficientnet_b

In [26]:
image = Image.from_path("images/peritonitis.jpg")

pipe_result = fgds_pipeline.predict(image, train_result=train_result)
pipe_result.outputs["classify"].class_index # Literally, peritonitis


2

In [27]:
# Don't forget to save the train result
train_result.export_artifacts("artifacts/fgds", fgds_pipeline)

In [28]:
# Also we can read these data from bundle:
fgds_pipeline.predict(image, bundle_path="artifacts/fgds")

PipelinePrediction(outputs={'transform': Image(body=tensor([[[2, 3, 3,  ..., 3, 2, 2],
         [2, 3, 4,  ..., 3, 3, 3],
         [3, 4, 4,  ..., 3, 2, 2],
         ...,
         [2, 2, 2,  ..., 1, 0, 0],
         [3, 3, 3,  ..., 3, 1, 1],
         [2, 3, 3,  ..., 0, 0, 0]],

        [[4, 5, 5,  ..., 3, 2, 2],
         [4, 5, 7,  ..., 3, 3, 3],
         [3, 3, 3,  ..., 3, 2, 2],
         ...,
         [2, 2, 2,  ..., 1, 0, 0],
         [3, 3, 3,  ..., 3, 1, 1],
         [2, 3, 3,  ..., 0, 0, 0]],

        [[1, 2, 4,  ..., 3, 2, 2],
         [1, 2, 6,  ..., 3, 3, 3],
         [1, 1, 4,  ..., 3, 2, 2],
         ...,
         [2, 2, 0,  ..., 1, 0, 0],
         [3, 4, 1,  ..., 3, 1, 1],
         [2, 3, 1,  ..., 0, 0, 0]]], dtype=torch.uint8), sample_id='peritonitis', legend=None), 'classify': ClassificationPrediction(logits=tensor([-3.4136,  1.1306,  3.1527, -2.6443], device='mps:0'), probs=tensor([0.0012, 0.1164, 0.8796, 0.0027], device='mps:0'), class_index=2, class_name=None)})